In [1]:
import numpy as np
import pandas as pd
from collections import Counter
import warnings, os, gc, re
import matplotlib.pyplot as plt
import psutil
import swifter
# import icd10
from ast import literal_eval

# from plotnine import ggplot, aes, geom_line
from plotnine import *
# import pygal as pg

%matplotlib inline

warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.4f}'.format

In [2]:
# count number of rows
from itertools import (takewhile,repeat)
def rawincount(filename):
    f = open(filename, 'rb')
    bufgen = takewhile(lambda x: x, (f.raw.read(1024*1024) for _ in repeat(None)))
    return sum( buf.count(b'\n') for buf in bufgen )

### Parm LIb declarations

In [3]:
path, dirs, files = next(os.walk("chunks_3/"))
directory = 'chunks_3/'
file_count = len(files)
year_start = 2013
year_end = 2020

In [4]:
# codes in ILLNESS*_CODE that are not ICD-10 codes
not_icd = ['MCP01', 'NSD01', 'P0000', 'P0001', 'ANC01', 'ANC02', 'FP001',
          'C19CI', 'C19CIS', 'C19FRP', 'C19IP1', 'C19IP2', 'C19IP3', 'C19IP4', 'C19T1', 'C19T2', 'C19T3', 'C19X3',
          'Z0011', 'Z0012', 'Z0013', 'Z0021', 'Z0022', 'Z003', 'Z0041', 'Z0042', 'Z0050', 'Z0051', 'Z0052', 'Z0061',
          'Z0062', 'Z0071', 'Z0072', 'Z0081', 'Z0082', 'Z0091', 'Z0092', 'Z010A', 'Z010B', 'Z010C', 'Z011A', 'Z011B',
          'Z011C', 'Z011D', 'Z011E', 'Z011F', 'Z011G', 'Z011G1', 'Z011G2', 'Z011H', 'Z011H1', 'Z011H2', 'Z011I',
          'Z01201', 'Z01202', 'Z01203', 'Z01204', 'Z01205', 'Z01206', 'Z01207', 'Z01208', 'Z01209', 'Z01210', 'Z01211',
          'Z01212', 'Z01213', 'Z01214', 'Z01215', 'Z01216', 'Z01217', 'Z01218', 'Z01219', 'Z01220', 'Z01221', 'Z01222',
          'Z01223', 'Z01224', 'Z01225', 'Z01226', 'Z0131A', 'Z0131B', 'Z0132B', 'Z0141A', 'Z0141B', 'Z0141CC', 'Z0141CL',
          'Z0142BC', 'Z0142BL', 'Z0142C', 'Z0143B', 'Z0143C', 'Z015101', 'Z0151A1', 'Z0151A2', 'Z0151B1', 'Z0151B2',
          'Z0151C1', 'Z0152A1', 'Z0152A2', 'Z0153A1', 'Z0153C1', 'Z0154A1', 'Z0154B1', 'Z0156A1', 'Z0156A2', 'Z0156B1',
          'Z0156B2', 'Z0156C1', 'Z0156C2', 'Z0157A1', 'Z0157B1', 'Z0157B2', 'Z0157C1', 'Z01591', 'Z0165', 'Z0166', 
          'Z0167', 'Z0168', 'Z0169']

#icd-10 philhealth list - 2017
df_icd10 = pd.read_excel(os.path.join(directory, 'ICD10 philhealth.xlsx'))
df_icd10 = df_icd10.set_axis(['ILLNESS1_CODE','DESCRIPTION','GROUP','CASE_RATE','PROFESSIONAL_FEE', \
                              'HEALTH_CARE_INST_FEE'], axis=1, inplace=False)

#procedures philhealth list - 2015
df_procs = pd.read_excel(os.path.join(directory, 'Procedures philhealth.xlsx'))
df_procs = df_procs.set_axis(['ILLNESS1_CODE','DESCRIPTION','CASE_RATE','PROFESSIONAL_FEE', \
                              'HEALTH_CARE_INST_FEE'], axis=1, inplace=False)

In [5]:
# 25th Percentile
def q25(x):
    return x.quantile(0.25)

# 75th Percentile
def q75(x):
    return x.quantile(0.75)

# counts per single categorical field
def group_categs(dframe, col_name_grp, ren_cols, tab_name, writer1):
    df2 = dframe.groupby([col_name_grp]).size().sort_values(ascending=False).to_frame().reset_index() \
                .set_axis([ren_cols, 'Freq.'], axis=1, inplace=False)
    df2['Percent'] = ((df2['Freq.'].values/total_year)*100)
    df2['Freq.'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
    df2['Percent'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name, index=False)
    del[df2]

# counts per combination of categorical fields
def sort_val_categs(dframe, col_name_grps, tab_name, writer1):
    de_bracket_cols = '[' + (', '.join('\'' + item + '\'' for item in col_name_grps)) + ', \'Freq.\']'
    df2 = df1.sort_values(col_name_grps,ascending=False).groupby(col_name_grps).size().to_frame().reset_index() \
            .set_axis(literal_eval(de_bracket_cols), axis=1, inplace=False)
    df2['Percent'] = ((df2['Freq.'].values/total_year)*100)
    df2['Freq.'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
    df2['Percent'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name, index=False)
    del[df2]

# computes for ILLNESS*_CODE total counts
def illness_counts(dframe, \
                   not_in_icd, df_icd, df_proc, \
                   col_name_grp, \
                   tab_name_med, tab_name_proc, \
                   writer1):   
    df2 = dframe.loc[dframe[col_name_grp].astype(str).str[0].str.isalpha() & ~dframe[col_name_grp].isin(not_in_icd)] \
            .groupby([col_name_grp]).size().sort_values(ascending=False).to_frame().reset_index() \
            .set_axis([col_name_grp, 'Freq.'], axis=1, inplace=False) \
            .merge(df_icd, how='outer', on=col_name_grp)[[col_name_grp,'Freq.','DESCRIPTION']]
    df2 = df2[~df2['Freq.'].isnull()]    
    df2['Percent'] = ((df2['Freq.'].values/total_year)*100)
    df2['Freq.'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
    df2['Percent'] = df2 = df2.loc[~df2['Freq.'].isnull()] 
    df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)        
    df2.to_excel(writer1, sheet_name=tab_name_med, index=False)
    del[df2]

#     df2 = dframe.loc[dframe[col_name_grp].str.isnumeric() | dframe[col_name_grp].isin(not_in_icd)] \
    df2 = dframe.loc[~(dframe[col_name_grp].astype(str).str[0].str.isalpha() & ~dframe[col_name_grp].isin(not_in_icd))] \
            .groupby([col_name_grp]).size().sort_values(ascending=False).to_frame().reset_index() \
            .set_axis([col_name_grp, 'Freq.'], axis=1, inplace=False) \
            .merge(df_proc, how='outer', on=col_name_grp)[[col_name_grp,'Freq.','DESCRIPTION']]
    df2 = df2[~df2['Freq.'].isnull()]      
    df2['Percent'] = ((df2['Freq.'].values/total_year)*100)
    df2['Freq.'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
    df2['Percent'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name_proc, index=False)
    del[df2]

# computes for ILLNESS*_CODE AMOUNT totals 
def illness_amts(dframe, \
                 not_in_icd, df_icd, df_proc, \
                 col_name_grp, amt_col, 
                 tab_name_med, tab_name_proc, \
                 tot_pd_amt, writer1): 
    df2 = dframe.loc[dframe[col_name_grp].astype(str).str[0].str.isalpha() & ~dframe[col_name_grp].isin(not_in_icd)] \
            .groupby([col_name_grp])[amt_col].sum().sort_values(ascending=False).to_frame().reset_index() \
            .set_axis([col_name_grp, 'Total Amount of Claims'], axis=1, inplace=False) \
            .merge(df_icd, how='outer', on=col_name_grp)[[col_name_grp,'Total Amount of Claims','DESCRIPTION']]  
    df2 = df2[~df2['Total Amount of Claims'].isnull()]      
    df2['Percent'] = ((df2['Total Amount of Claims'].values/tot_pd_amt)*100)
    df2['Total Amount of Claims'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}' \
                                        .format(x['Total Amount of Claims']),axis=1)
    df2['Percent'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name_med, index=False)
    del[df2]
        
#     df2 = dframe.loc[dframe[col_name_grp].str.isnumeric() | dframe[col_name_grp].isin(not_in_icd)] \
    df2 = dframe.loc[~(dframe[col_name_grp].astype(str).str[0].str.isalpha() & ~dframe[col_name_grp].isin(not_in_icd))] \
            .groupby([col_name_grp])[amt_col].sum().sort_values(ascending=False).to_frame().reset_index() \
            .set_axis([col_name_grp, 'Total Amount of Claims'], axis=1, inplace=False) \
            .merge(df_proc, how='outer', on=col_name_grp)[[col_name_grp,'Total Amount of Claims','DESCRIPTION']]    
    df2 = df2[~df2['Total Amount of Claims'].isnull()]       
    df2['Percent'] = ((df2['Total Amount of Claims'].values/tot_pd_amt)*100)
    df2['Total Amount of Claims'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}' \
                                        .format(x['Total Amount of Claims']),axis=1)
    df2['Percent'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name_proc, index=False)
    del[df2]

# computes for ILLNESS*_CODE totals, +1 col grouping besides illness code field
def illness_counts_oth_1(dframe, 
                not_in_icd, df_icd, df_proc, \
                illness_cd_col, col_name_grp, \
                tab_name_med, tab_name_proc, \
                tot_year, writer1):
    df2 = df1.loc[df1[illness_cd_col].astype(str).str[0].str.isalpha() & ~df1[illness_cd_col].isin(not_icd)] \
            .groupby([col_name_grp, illness_cd_col]).size().to_frame().reset_index() \
            .set_axis([col_name_grp, illness_cd_col, 'Freq.'], axis=1, inplace=False) \
            .merge(df_icd, how='outer', on=illness_cd_col)[[col_name_grp, illness_cd_col, \
                                                                'Freq.','DESCRIPTION']]  \
            .sort_values(by=[col_name_grp, 'Freq.'], ascending=[True, False])
    df2 = df2[~df2['Freq.'].isnull()]       
    df2['Percent'] = ((df2['Freq.']/tot_year)*100)
    df2['Freq.'] = df2.apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
    df2['Percent'] = df2.apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)        
    df2.to_excel(writer1, sheet_name=tab_name_med, index=False)
    del[df2]

#     df2 = df1.loc[df1[illness_cd_col].str.isnumeric() | df1[illness_cd_col].isin(not_icd)] \
    df2 = df1.loc[~(df1[illness_cd_col].astype(str).str[0].str.isalpha() & ~df1[illness_cd_col].isin(not_icd))] \
            .groupby([col_name_grp, illness_cd_col]).size().to_frame().reset_index() \
            .set_axis([col_name_grp,illness_cd_col, 'Freq.'], axis=1, inplace=False) \
            .merge(df_proc, how='outer', on=illness_cd_col)[[col_name_grp, illness_cd_col, \
                                                                'Freq.','DESCRIPTION']] \
            .sort_values(by=[col_name_grp, 'Freq.'], ascending=[True, False])
    df2 = df2[~df2['Freq.'].isnull()]      
    df2['Percent'] = ((df2['Freq.']/tot_year)*100)
    df2['Freq.'] = df2.apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
    df2['Percent'] = df2.apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name_proc, index=False)
    del[df2]

# computes for ILLNESS*_CODE amounts, +1 col grouping besides illness code field    
def illness_amts_oth_1(dframe, 
                not_in_icd, df_icd, df_proc, \
                illness_cd_col, col_name_grp, amt_col, \
                tab_name_med, tab_name_proc, \
                tot_amt, writer1):    
    df2 = df1.loc[df1[illness_cd_col].astype(str).str[0].str.isalpha() & ~df1[illness_cd_col].isin(not_icd)] \
            .groupby([col_name_grp, illness_cd_col])[amt_col].sum().to_frame().reset_index() \
            .set_axis([col_name_grp, illness_cd_col, 'Total Amount of Claims'], axis=1, inplace=False) \
            .merge(df_icd, how='outer', on=illness_cd_col)[[col_name_grp, illness_cd_col, \
                                                                'Total Amount of Claims','DESCRIPTION']] \
            .sort_values(by=[col_name_grp, 'Total Amount of Claims'], ascending=[True, False])
    df2 = df2[~df2['Total Amount of Claims'].isnull()]        
    df2['Percent'] = ((df2['Total Amount of Claims']/tot_amt)*100)
    df2['Total Amount of Claims'] = df2.apply(lambda x: '{:,.2f}'.format(x['Total Amount of Claims']),axis=1)
    df2['Percent'] = df2.apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name_med, index=False)
    del[df2]

#     df2 = df1.loc[df1[illness_cd_col].str.isnumeric() | df1[illness_cd_col].isin(not_icd)] \
    df2 = df1.loc[~(df1[illness_cd_col].astype(str).str[0].str.isalpha() & ~df1[illness_cd_col].isin(not_icd))] \
            .groupby([col_name_grp, illness_cd_col])[amt_col].sum().to_frame().reset_index() \
            .set_axis([col_name_grp, illness_cd_col, 'Total Amount of Claims'], axis=1, inplace=False) \
            .merge(df_proc, how='outer', on=illness_cd_col)[[col_name_grp, illness_cd_col, \
                                                                'Total Amount of Claims','DESCRIPTION']] \
            .sort_values(by=[col_name_grp, 'Total Amount of Claims'], ascending=[True, False])
    df2 = df2[~df2['Total Amount of Claims'].isnull()]      
    df2['Percent'] = ((df2['Total Amount of Claims']/tot_amt)*100)
    df2['Total Amount of Claims'] = df2.apply(lambda x: '{:,.2f}'.format(x['Total Amount of Claims']),axis=1)
    df2['Percent'] = df2.apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name_proc, index=False)
    del[df2]

def turnaround_time_1(dframe, col_name_grp, tab_name, tab_name_365):   
    df2 = df1.loc[df1['TAT'] >= 0].groupby([col_name_grp])['TAT'] \
            .agg(['mean', 'median', 'std', 'min', 'max', q25, q75, 'count']) 
    df3 = df1.loc[(df1['TAT'] > 60)].groupby([col_name_grp]).size().to_frame().reset_index()
    df3 = df3.set_axis([col_name_grp, 'numgreater60'], axis=1, inplace=False)
    df2 = df2.merge(df3, how='inner', on=col_name_grp)
    df2['%>60days'] = 100*(df2['numgreater60']/df2['count'])
    df2.to_excel(writer, sheet_name=tab_name, index=False)
    del[df2]
    del[df3]

    df2 = df1.loc[(df1['TAT'] >= 0) & (df1['TAT'] <= 365)].groupby([col_name_grp])['TAT'] \
            .agg(['mean', 'median', 'std', 'min', 'max', q25, q75, 'count']) 
    df3 = df1.loc[(df1['TAT'] > 60) & (df1['TAT'] <= 365)].groupby([col_name_grp]).size().to_frame().reset_index()
    df3 = df3.set_axis([col_name_grp, 'numgreater60'], axis=1, inplace=False)
    df2 = df2.merge(df3, how='inner', on=col_name_grp)
    df2['%>60days'] = 100*(df2['numgreater60']/df2['count'])
    df2.to_excel(writer, sheet_name=tab_name_365, index=False)
    del[df2]
    del[df3]
    
def unpaid_process(dframe, \
                check_dt_col, \
                rr_col, amt_col, acr_amt_col, \
                agg_col, tab_name, writer1):
#     df2 = dframe[dframe[check_dt_col].isna() & ~dframe[rr_col].isna()]
    df2 = dframe[dframe['CLAIM_STATUS'] == check_dt_col]
    df2[amt_col].fillna(df2[acr_amt_col], inplace=True)    
    df2.groupby([agg_col])[amt_col].agg(['mean', 'median', 'count', 'min', 'max']) \
        .to_excel(writer1, sheet_name=tab_name, index=True)
    del[df2]

In [15]:
for i in range(year_start, year_end+1):
    df_all = pd.read_csv(os.path.join(directory + 'UP-' + str(i) + '.csv'), low_memory=False, index_col=0)
    df1 = df_all[(df_all['CLAIM_STATUS'] == 'PAID') | (df_all['CLAIM_STATUS'] == 'APPROVED FOR PAYMENT')] \
            .drop(columns=['SERIES','MEMBER_NAME','PATIENT_NAME'])

    # for lighter pandas df and faster execution
    df1['PRO_NAME'] = df1['PRO_NAME'].astype('category')
    df1['INSTITUTION_NAME'] = df1['INSTITUTION_NAME'].astype('category')
    df1['OWNERSHIP'] = df1['OWNERSHIP'].astype('category')
    df1['INSTITUTION_CLASS'] = df1['INSTITUTION_CLASS'].astype('category')
    df1['INSTITUTION_PROVINCE'] = df1['INSTITUTION_PROVINCE'].astype('category')
    df1['INSTITUTION_MUNICIPALITY'] = df1['INSTITUTION_MUNICIPALITY'].astype('category')
    df1['MEM_CATEGORY'] = df1['MEM_CATEGORY'].astype('category')
#     df1['MEM_SUB_CATEGORY'] = df1['MEM_SUB_CATEGORY'].astype('category')

#     # computing for Turnaround Time
#     df1[['CHECK_DT','RECEIVED/REFILED DATE']] = df1[['CHECK_DT','RECEIVED/REFILED DATE']].apply(pd.to_datetime)
#     df1['TAT'] = (df1['CHECK_DT'] - df1['RECEIVED/REFILED DATE']).dt.days

#     # computing for Financial Coverage; set x/0 row as NaN so it will not be included in the aggregates
#     df1['FINANCIAL_COVERAGE'] = df1['PAID AMOUNT']/df1['ACTUAL_AMT']
#     df1.loc[~np.isfinite(df1['FINANCIAL_COVERAGE']), 'FINANCIAL_COVERAGE'] = np.nan

    print(i)
    print(len(df_all.index))
    total_year = len(df1.index)
    total_paid_amt = df1['PAID AMOUNT'].sum()
#     print('with illness code NaN:', len(df1[df1['ILLNESS1_CODE']].isnull()))
    
    writer = pd.ExcelWriter(os.path.join(directory, str(i) + '_totals.xlsx'), engine='xlsxwriter')

#     group_categs(df1,'PRO_NAME','region','No. of Claims by Region', writer)
#     group_categs(df1,'OWNERSHIP','OWNERSHIP','No. of Claims by Hosp Owner', writer)
#     group_categs(df1,'INSTITUTION_CLASS','INSTITUTION_CLASS','No. of Claims by Inst Class', writer)
#     group_categs(df1,'INSTITUTION_PROVINCE','INSTITUTION_PROVINCE','No. of Claims by Inst Prov', writer)
        
#     sort_val_categs(df1, ['MEM_CATEGORY','MEM_SUB_CATEGORY'], 'Membership Overall', writer)
#     sort_val_categs(df1, ['PRO_NAME','MEM_SUB_CATEGORY'], 'Membership by Region', writer)
#     sort_val_categs(df1, ['MEM_CATEGORY','INSTITUTION_CLASS'], 'Membership by Inst Class', writer)
#     sort_val_categs(df1, ['MEM_CATEGORY','OWNERSHIP'], 'Membership by Ownership', writer)

#     illness_counts(df1, not_icd, df_icd10, df_procs, 'ILLNESS1_CODE', \
#                    'Med Claim - Counts', 'Proc Claim - Counts', writer)
#     illness_amts(df1, not_icd, df_icd10, df_procs, 'ILLNESS1_CODE', 'PAID AMOUNT', \
#                  'Med Claim - Sum', 'Proc Claim - Sum', total_paid_amt, writer)
        
#     turnaround_time_1(df1, 'INSTITUTION_CLASS', 'TAT by Inst Class', 'TAT by Inst Class-365 days max')
#     turnaround_time_1(df1, 'PRO_NAME', 'TAT by Region', 'TAT by Region-365 days max')
        
#     df1.groupby(['MEM_CATEGORY'])['FINANCIAL_COVERAGE'].agg(['mean', 'median', 'count', 'min', 'max']) \
#                             .set_axis(['MEM_CATEGORY', 'Freq.'], axis=1, inplace=False) \
#                             .to_excel(writer, sheet_name='Supp Val by Membership')  
        
#     df1.groupby(['INSTITUTION_CLASS'])['FINANCIAL_COVERAGE'].agg(['mean', 'median', 'count', 'min', 'max']) \
#                             .set_axis(['INSTITUTION_CLASS', 'Freq.'], axis=1, inplace=False) \    
#                             .to_excel(writer, sheet_name='Supp Val by Inst Class') 
        
#     df1.groupby(['OWNERSHIP'])['FINANCIAL_COVERAGE'].agg(['mean', 'median', 'count', 'min', 'max']) \
#                             .set_axis(['OWNERSHIP', 'Freq.'], axis=1, inplace=False) \        
#                             .to_excel(writer, sheet_name='Supp Val by Owner')  
        
#     df1.groupby(['PRO_NAME'])['FINANCIAL_COVERAGE'].agg(['mean', 'median', 'count', 'min', 'max']) \
#                             .set_axis(['PRO_NAME', 'Freq.'], axis=1, inplace=False) \            
#                             .to_excel(writer, sheet_name='Supp Val by Region') 

#     illness_counts_oth_1(df1, 
#                     not_icd, df_icd10, df_procs, \
#                     'ILLNESS1_CODE', 'INSTITUTION_CLASS', \
#                     'Med Claim - Counts-Inst Class', 'Proc Claim - Counts-Inst Class', \
#                     total_year, writer)    

#     illness_amts_oth_1(df1, 
#                     not_icd, df_icd10, df_procs, \
#                     'ILLNESS1_CODE', 'INSTITUTION_CLASS', 'PAID AMOUNT', \
#                     'Med Claim - Amt-Inst Class', 'Proc Claim - Amt-Inst Class', \
#                     total_paid_amt, writer)

#     illness_counts_oth_1(df1, 
#                     not_icd, df_icd10, df_procs, \
#                     'ILLNESS1_CODE', 'PRO_NAME', \
#                     'Med Claim - Counts-Region', 'Proc Claim - Counts-Region', \
#                     total_year, writer)    

#     illness_amts_oth_1(df1, 
#                     not_icd, df_icd10, df_procs, \
#                     'ILLNESS1_CODE', 'PRO_NAME', 'PAID AMOUNT', \
#                     'Med Claim - Amt-Region', 'Proc Claim - Amt-Region', \
#                     total_paid_amt, writer)

#     illness_counts_oth_1(df1, 
#                     not_icd, df_icd10, df_procs, \
#                     'ILLNESS1_CODE', 'INSTITUTION_PROVINCE', \
#                     'Med Claim - Counts-Province', 'Proc Claim - Counts-Province', \
#                     total_year, writer)    

#     illness_amts_oth_1(df1, 
#                     not_icd, df_icd10, df_procs, \
#                     'ILLNESS1_CODE', 'INSTITUTION_PROVINCE', 'PAID AMOUNT', \
#                     'Med Claim - Amt-Province', 'Proc Claim - Amt-Province', \
#                     total_paid_amt, writer)

#     illness_counts_oth_1(df1, 
#                     not_icd, df_icd10, df_procs, \
#                     'ILLNESS1_CODE', 'OWNERSHIP', \
#                     'Med Claim - Counts-Ownership', 'Proc Claim - Counts-Ownership', \
#                     total_year, writer)    

#     illness_amts_oth_1(df1, 
#                     not_icd, df_icd10, df_procs, \
#                     'ILLNESS1_CODE', 'OWNERSHIP', 'PAID AMOUNT', \
#                     'Med Claim - Amt-Ownership', 'Proc Claim - Amt-Ownership', \
#                     total_paid_amt, writer)
    
#     ## Unpaid Claims processing
#     df_unpaid = df_all[(df_all['CLAIM_STATUS'] != 'PAID') & (df_all['CLAIM_STATUS'] != 'APPROVED FOR PAYMENT')] \
#             .drop(columns=['SERIES','MEMBER_NAME','PATIENT_NAME'])

#     df_unpaid['PRO_NAME'] = df_unpaid['PRO_NAME'].astype('category')
#     df_unpaid['INSTITUTION_NAME'] = df_unpaid['INSTITUTION_NAME'].astype('category')
#     df_unpaid['OWNERSHIP'] = df_unpaid['OWNERSHIP'].astype('category')
#     df_unpaid['INSTITUTION_CLASS'] = df_unpaid['INSTITUTION_CLASS'].astype('category')
#     df_unpaid['INSTITUTION_PROVINCE'] = df_unpaid['INSTITUTION_PROVINCE'].astype('category')
#     df_unpaid['INSTITUTION_MUNICIPALITY'] = df_unpaid['INSTITUTION_MUNICIPALITY'].astype('category')
#     df_unpaid['MEM_CATEGORY'] = df_unpaid['MEM_CATEGORY'].astype('category')
    
#     df_unpaid.groupby(['INSTITUTION_CLASS']).size().sort_values(ascending=False).to_frame().reset_index() \
#                             .set_axis(['INSTITUTION_CLASS', 'Freq.'], axis=1, inplace=False) \
#                             .to_excel(writer, sheet_name='Unpaid Clms - by Inst Class')
        
#     df_unpaid.groupby(['MEM_CATEGORY']).size().sort_values(ascending=False).to_frame().reset_index() \
#                             .set_axis(['MEM_CATEGORY', 'Freq.'], axis=1, inplace=False) \
#                             .to_excel(writer, sheet_name='Unpaid Clms - by Membership')
        
#     df_unpaid.groupby(['PRO_NAME']).size().sort_values(ascending=False).to_frame().reset_index() \
#                             .set_axis(['PRO_NAME', 'Freq.'], axis=1, inplace=False) \
#                             .to_excel(writer, sheet_name='Unpaid Clms - by Region')    

#     illness_counts(df_unpaid, not_icd, df_icd10, df_procs, 'ILLNESS1_CODE', \
#                    'Unpd Med Claim - Counts', 'Unpd Proc Claim - Counts', writer)
    
#     illness_amts(df_unpaid, not_icd, df_icd10, df_procs, 'ILLNESS1_CODE', 'PAID AMOUNT', \
#                  'Unpd Med Claim - Sum', 'Unpd Proc Claim - Sum', total_paid_amt, writer)

#     df_unpaid.groupby(['CLAIM_STATUS', 'INSTITUTION_CLASS']).size().sort_values(ascending=False).to_frame().reset_index() \
#                             .set_axis(['CLAIM_STATUS', 'INSTITUTION_CLASS', 'Freq.'], axis=1, inplace=False) \
#                             .to_excel(writer, sheet_name='Unpaid Clms - by Inst Class')
        
#     df_unpaid.groupby(['CLAIM_STATUS', 'MEM_CATEGORY']).size().sort_values(ascending=False).to_frame().reset_index() \
#                             .set_axis(['CLAIM_STATUS', 'MEM_CATEGORY', 'Freq.'], axis=1, inplace=False) \
#                             .to_excel(writer, sheet_name='Unpaid Clms - by Membership')
        
#     df_unpaid.groupby(['CLAIM_STATUS', 'PRO_NAME']).size().sort_values(ascending=False).to_frame().reset_index() \
#                             .set_axis(['CLAIM_STATUS', 'PRO_NAME', 'Freq.'], axis=1, inplace=False) \
#                             .to_excel(writer, sheet_name='Unpaid Clms - by Region')    

#     unpaid_process(df_unpaid, 
#                     'RTH', 'RECEIVED/REFILED DATE', 'ACTUAL_AMT', 'ILLNESS_1_ACR_AMT', \
#                     'PRO_NAME', 'PRO-Unpd Clms ACTUAL_AMT-RTH', writer)

#     unpaid_process(df_unpaid, 
#                     'RTH', 'RECEIVED/REFILED DATE', 'ACTUAL_AMT', 'ILLNESS_1_ACR_AMT', \
#                     'INSTITUTION_CLASS', 'INSTC-Unpd Clms ACTUAL_AMT-RTH', writer)

#     unpaid_process(df_unpaid, 
#                     'IN-PROCESS', 'RECEIVED/REFILED DATE', 'ACTUAL_AMT', 'ILLNESS_1_ACR_AMT', \
#                     'PRO_NAME', 'PRO-Unpd Clms ACTUAL_AMT-INP', writer)

#     unpaid_process(df_unpaid, 
#                     'IN-PROCESS', 'RECEIVED/REFILED DATE', 'ACTUAL_AMT', 'ILLNESS_1_ACR_AMT', \
#                     'INSTITUTION_CLASS', 'INSTC-Unpd Clms ACTUAL_AMT-INP', writer)

#     unpaid_process(df_unpaid, 
#                     'DENIED', 'RECEIVED/REFILED DATE', 'ACTUAL_AMT', 'ILLNESS_1_ACR_AMT', \
#                     'PRO_NAME', 'PRO-Unpd Clms ACTUAL_AMT-DND', writer)

#     unpaid_process(df_unpaid, 
#                     'DENIED', 'RECEIVED/REFILED DATE', 'ACTUAL_AMT', 'ILLNESS_1_ACR_AMT', \
#                     'INSTITUTION_CLASS', 'INSTC-Unpd Clms ACTUAL_AMT-DND', writer)
    
#     unpaid_process(df_unpaid, 
#                     'CHECK_DT', 'RECEIVED/REFILED DATE', 'ACTUAL_AMT', 'ILLNESS_1_ACR_AMT', \
#                     'PRO_NAME', 'Unpd Clms ACTUAL_AMT-Region', writer)

#     unpaid_process(df_unpaid, 
#                     'CHECK_DT', 'RECEIVED/REFILED DATE', 'ACTUAL_AMT', 'ILLNESS_1_ACR_AMT', \
#                     'INSTITUTION_CLASS', 'Unpd Clms ACTUAL_AMT-Inst Cls', writer)

#     unpaid_process(df_unpaid, 
#                     'CHECK_DT', 'RECEIVED_REFILED_DATE', 'ACTUAL_AMT', 'ILLNESS_1_ACR_AMT', \
#                     'MEM_SUB_CATEGORY', 'Unpd Clms ACTUAL_AMT-Sub Categ', writer)
    
    writer.save()  
   
    del[df1]
#     del[df_unpaid]
    del[df_all]
    gc.collect()
    gc.collect()
    gc.collect()

2018
11562595
2019
11744401
2020
11226777


In [6]:
## creating top 10 medical claims
df_j1892 = pd.DataFrame()
df_a099 = pd.DataFrame()
df_n390 = pd.DataFrame()
df_j4590 = pd.DataFrame()
df_k291 = pd.DataFrame()
df_i101 = pd.DataFrame()
df_i109 = pd.DataFrame()
df_n180 = pd.DataFrame()
df_p369 = pd.DataFrame()
df_a971 = pd.DataFrame()
df_a911 = pd.DataFrame()

In [7]:
## creating top 10 medical claims
for i in range(year_start, year_end+1):
    df_all = pd.read_csv(os.path.join(directory + 'UP-' + str(i) + '.csv'), low_memory=False, index_col=0)
    df1 = df_all[(df_all['CLAIM_STATUS'] == 'PAID') | (df_all['CLAIM_STATUS'] == 'APPROVED FOR PAYMENT')] \
            .drop(columns=['SERIES','MEMBER_NAME','PATIENT_NAME'])
#             .head(20000)

    # for lighter pandas df and faster execution
    df1['PRO_NAME'] = df1['PRO_NAME'].astype('category')
    df1['INSTITUTION_NAME'] = df1['INSTITUTION_NAME'].astype('category')
    df1['OWNERSHIP'] = df1['OWNERSHIP'].astype('category')
    df1['INSTITUTION_CLASS'] = df1['INSTITUTION_CLASS'].astype('category')
    df1['INSTITUTION_PROVINCE'] = df1['INSTITUTION_PROVINCE'].astype('category')
    df1['INSTITUTION_MUNICIPALITY'] = df1['INSTITUTION_MUNICIPALITY'].astype('category')
    df1['MEM_CATEGORY'] = df1['MEM_CATEGORY'].astype('category')
    df1.insert(0, 'YEAR', str(i))
    
    del[df_all]
    gc.collect()
    gc.collect()    

    print(i)
    print(len(df1))
    
    df_j1892 = df_j1892.append(df1[(df1['ILLNESS1_CODE'] == 'J18.92')]).reset_index(drop=True)                                                                                    
    df_a099  = df_a099.append(df1[(df1['ILLNESS1_CODE'] == 'A09.9')]).reset_index(drop=True)
    df_n390  = df_n390.append(df1[(df1['ILLNESS1_CODE'] == 'N39.0')]).reset_index(drop=True)
    df_j4590 = df_j4590.append(df1[(df1['ILLNESS1_CODE'] == 'J45.90')]).reset_index(drop=True)
    df_k291  = df_k291.append(df1[(df1['ILLNESS1_CODE'] == 'K29.1')]).reset_index(drop=True)
    df_i101  = df_i101.append(df1[(df1['ILLNESS1_CODE'] == 'I10.1')]).reset_index(drop=True)
    df_i109  = df_i109.append(df1[(df1['ILLNESS1_CODE'] == 'I10.9')]).reset_index(drop=True)
    df_n180  = df_n180.append(df1[(df1['ILLNESS1_CODE'] == 'N18.0')]).reset_index(drop=True)
    df_p369  = df_p369.append(df1[(df1['ILLNESS1_CODE'] == 'P36.9')]).reset_index(drop=True)
    df_a971  = df_a971.append(df1[(df1['ILLNESS1_CODE'] == 'A97.1')]).reset_index(drop=True)
    df_a911  = df_a911.append(df1[(df1['ILLNESS1_CODE'] == 'A91.1')]).reset_index(drop=True)
    
    del[df1]
    gc.collect()
    gc.collect()
    gc.collect()    

2013
5952937
2014
6624993
2015
8750269
2016
10320740
2017
10472915
2018
10813679
2019
10825169
2020
10181139


In [8]:
## creating top 10 medical claims
df_j1892.to_csv('J18_92.csv', index=False)
print('J18.92',len(df_j1892))

df_a099.to_csv('A09_9.csv', index=False)
print('A09.9',len(df_a099))

df_n390.to_csv('N39_0.csv', index=False)
print('N39.0',len(df_n390))

df_j4590.to_csv('J45_90.csv', index=False)
print('J45.90',len(df_j4590))

df_k291.to_csv('K29_1.csv', index=False)
print('K29.1',len(df_k291))

df_i101.to_csv('I10_1.csv', index=False)
print('I10.1',len(df_i101))

df_i109.to_csv('I10_9.csv', index=False)
print('I10.9',len(df_i109))

df_n180.to_csv('N18_0.csv', index=False)
print('N18.0',len(df_n180))

df_p369.to_csv('P36_9.csv', index=False)
print('P36.9',len(df_p369))

df_a971.to_csv('A97_1.csv', index=False)
print('A97.1',len(df_a971))

df_a911.to_csv('A91_1.csv', index=False)
print('A91.1',len(df_a911))

J18.92 5018427
A09.9 2230538
N39.0 1983352
J45.90 883032
K29.1 803247
I10.1 774396
I10.9 766362
N18.0 655275
P36.9 603329
A97.1 580687
A91.1 525029


In [10]:
del[df_j1892]                                                                                   
del[df_a099]
del[df_n390]
del[df_j4590]
del[df_k291]
del[df_i101]
del[df_i109]
del[df_n180]
del[df_p369]
del[df_a971]
del[df_a911]
gc.collect()
gc.collect()
gc.collect()    

0

In [11]:
## creating top 10 procedure claims
df_90935 = pd.DataFrame()
df_99432 = pd.DataFrame()
df_NSD01 = pd.DataFrame()
df_MCP01 = pd.DataFrame()
df_99460 = pd.DataFrame()
df_59513 = pd.DataFrame()
df_96408 = pd.DataFrame()
df_59409 = pd.DataFrame()
df_59514 = pd.DataFrame()
df_66987 = pd.DataFrame()
df_77401 = pd.DataFrame()
df_58120 = pd.DataFrame()

In [12]:
## creating top 10 procedure claims
for i in range(year_start, year_end+1):
    df_all = pd.read_csv(os.path.join(directory + 'UP-' + str(i) + '.csv'), low_memory=False, index_col=0)
    df1 = df_all[(df_all['CLAIM_STATUS'] == 'PAID') | (df_all['CLAIM_STATUS'] == 'APPROVED FOR PAYMENT')] \
            .drop(columns=['SERIES','MEMBER_NAME','PATIENT_NAME'])
#             .head(20000)

    # for lighter pandas df and faster execution
    df1['PRO_NAME'] = df1['PRO_NAME'].astype('category')
    df1['INSTITUTION_NAME'] = df1['INSTITUTION_NAME'].astype('category')
    df1['OWNERSHIP'] = df1['OWNERSHIP'].astype('category')
    df1['INSTITUTION_CLASS'] = df1['INSTITUTION_CLASS'].astype('category')
    df1['INSTITUTION_PROVINCE'] = df1['INSTITUTION_PROVINCE'].astype('category')
    df1['INSTITUTION_MUNICIPALITY'] = df1['INSTITUTION_MUNICIPALITY'].astype('category')
    df1['MEM_CATEGORY'] = df1['MEM_CATEGORY'].astype('category')
    df1.insert(0, 'YEAR', str(i))
    
    del[df_all]
    gc.collect()
    gc.collect()    

    print(i)
    print(len(df1))
    
    df_90935 = df_90935.append(df1[(df1['ILLNESS1_CODE'] == '90935')]).reset_index(drop=True)                                                                                    
    df_99432 = df_99432.append(df1[(df1['ILLNESS1_CODE'] == '99432')]).reset_index(drop=True) 
    df_NSD01 = df_NSD01.append(df1[(df1['ILLNESS1_CODE'] == 'NSD01')]).reset_index(drop=True) 
    df_MCP01 = df_MCP01.append(df1[(df1['ILLNESS1_CODE'] == 'MCP01')]).reset_index(drop=True) 
    df_99460 = df_99460.append(df1[(df1['ILLNESS1_CODE'] == '99460')]).reset_index(drop=True) 
    df_59513 = df_59513.append(df1[(df1['ILLNESS1_CODE'] == '59513')]).reset_index(drop=True) 
    df_96408 = df_96408.append(df1[(df1['ILLNESS1_CODE'] == '96408')]).reset_index(drop=True) 
    df_59409 = df_59409.append(df1[(df1['ILLNESS1_CODE'] == '59409')]).reset_index(drop=True) 
    df_59514 = df_59514.append(df1[(df1['ILLNESS1_CODE'] == '59514')]).reset_index(drop=True) 
    df_66987 = df_66987.append(df1[(df1['ILLNESS1_CODE'] == '66987')]).reset_index(drop=True) 
    df_77401 = df_77401.append(df1[(df1['ILLNESS1_CODE'] == '77401')]).reset_index(drop=True) 
    df_58120 = df_58120.append(df1[(df1['ILLNESS1_CODE'] == '58120')]).reset_index(drop=True) 
    
    del[df1]
    gc.collect()
    gc.collect()
    gc.collect()    

2013
5952937
2014
6624993
2015
8750269
2016
10320740
2017
10472915
2018
10813679
2019
10825169
2020
10181139


In [13]:
## creating top 10 procedure claims
df_90935.to_csv('p90935.csv', index=False)
print('90935',len(df_90935))

df_99432.to_csv('p99432.csv', index=False)
print('99432',len(df_99432))

df_NSD01.to_csv('NSD01.csv', index=False)
print('NSD01',len(df_NSD01))

df_MCP01.to_csv('MCP01.csv', index=False)
print('MCP01',len(df_MCP01))

df_99460.to_csv('p99460.csv', index=False)
print('99460',len(df_99460))

df_59513.to_csv('p59513.csv', index=False)
print('59513',len(df_59513))

df_96408.to_csv('p96408.csv', index=False)
print('96408',len(df_96408))

df_59409.to_csv('p59409.csv', index=False)
print('59409',len(df_59409))

df_59514.to_csv('p59514.csv', index=False)
print('59514',len(df_59514))

df_66987.to_csv('p66987.csv', index=False)
print('66987',len(df_66987))

df_77401.to_csv('p77401.csv', index=False)
print('77401',len(df_77401))

df_58120.to_csv('p58120.csv', index=False)
print('58120',len(df_58120))

90935 11988992
99432 4150748
NSD01 2806238
MCP01 1972131
99460 1345360
59513 1154665
96408 925538
59409 909762
59514 822261
66987 742169
77401 696447
58120 382889


In [15]:
del[df_90935]
del[df_99432]
del[df_NSD01]
del[df_MCP01]
del[df_99460]
del[df_59513] 
del[df_96408]
del[df_59409]
del[df_59514]
del[df_66987]
del[df_77401]
del[df_58120]
gc.collect()
gc.collect()
gc.collect()  

0

### Fuzzy Matching for Hospital Names

In [7]:
from rapidfuzz import process, utils

In [8]:
# import pandas as pd, numpy as np

for i in range(year_start, year_end+1):
    df1 = pd.read_csv(os.path.join(directory + 'UP-' + str(i) + '.csv'), low_memory=False, index_col=0)
    df2 = pd.DataFrame(df1['INSTITUTION_NAME'].unique(), columns=['INSTITUTION_NAME'])
    org_list = df2['INSTITUTION_NAME'].unique().astype(str)
    processed_orgs = [utils.default_process(org) for org in org_list]
    
    writer = pd.ExcelWriter(os.path.join(directory, str(i) + '_fuzzy_match_inst_name.xlsx'), engine='xlsxwriter')
    
    for (i, processed_query) in enumerate(processed_orgs):
        processed_orgs[i] = None
        match = process.extractOne(processed_query, processed_orgs, processor=None, score_cutoff=95)
        processed_orgs[i] = processed_query
        if match:
            df2.loc[i, 'fuzzy_match'] = org_list[match[2]]
            df2.loc[i, 'fuzzy_match_score'] = match[1]

    df3 = df2[(df2['fuzzy_match'] > '')].drop(columns=['fuzzy_match_score'])
    df3.to_excel(writer, sheet_name='Fuzzy Match', index=False)
    
    writer.save()
    
    del[df1]
    del[df2]
    del[df3]
    gc.collect()
    gc.collect()    